In [1]:
# =========================================
# Cell 1 — Install dependencies
# =========================================
!pip -q install datasets>=2.19.0 openai>=1.0.0 pandas>=2.2.0 numpy>=1.26.0 scikit-learn>=1.3.0 tqdm>=4.66.0 matplotlib>=3.8.0


In [2]:
# =========================================
# Cell 2 — Keys (type/paste in)
# =========================================
import os, getpass

os.environ["OPENAI_API_KEY"] = getpass.getpass("Enter OPENAI_API_KEY (hidden): ")

hf = getpass.getpass("Enter HF_TOKEN (optional; press Enter to skip): ")
if hf.strip():
    os.environ["HF_TOKEN"] = hf.strip()

assert os.environ.get("OPENAI_API_KEY"), "Missing OPENAI_API_KEY"
print("✅ Keys set. HF token provided?", bool(os.environ.get("HF_TOKEN")))


Enter OPENAI_API_KEY (hidden): ··········
Enter HF_TOKEN (optional; press Enter to skip): ··········
✅ Keys set. HF token provided? True


In [3]:
# =========================================
# Cell 3 — Load flare-fiqasa from Hugging Face
# =========================================
from datasets import load_dataset

DATASET_NAME = "TheFinAI/flare-fiqasa"
SPLIT = "test"   # choose: "train", "valid", or "test"

ds = load_dataset(DATASET_NAME, split=SPLIT, token=os.environ.get("HF_TOKEN", None))
print(ds)
print("Columns:", ds.column_names)
print("Example row:\n", ds[0])


README.md:   0%|          | 0.00/1.13k [00:00<?, ?B/s]

data/train-00000-of-00001-d0f9b6513e12e0(…):   0%|          | 0.00/100k [00:00<?, ?B/s]

data/test-00000-of-00001-faca082021057ac(…):   0%|          | 0.00/35.8k [00:00<?, ?B/s]

data/valid-00000-of-00001-36997935dc03cb(…):   0%|          | 0.00/29.9k [00:00<?, ?B/s]

Generating train split:   0%|          | 0/750 [00:00<?, ? examples/s]

Generating test split:   0%|          | 0/235 [00:00<?, ? examples/s]

Generating valid split:   0%|          | 0/188 [00:00<?, ? examples/s]

Dataset({
    features: ['id', 'query', 'answer', 'text', 'choices', 'gold'],
    num_rows: 235
})
Columns: ['id', 'query', 'answer', 'text', 'choices', 'gold']
Example row:
 {'id': 'fiqasa938', 'query': 'What is the sentiment of the following financial post: Positive, Negative, or Neutral?\nText: $BBRY Actually lost .03c per share if U incl VZ as no debt and 3.1 in Cash.\nAnswer:', 'answer': 'negative', 'text': '$BBRY Actually lost .03c per share if U incl VZ as no debt and 3.1 in Cash.', 'choices': ['negative', 'positive', 'neutral'], 'gold': 0}


In [4]:
# =========================================
# Cell 4 — GPT-4.1 (GPT-4 family): prompt + API helpers (Responses API)
# =========================================
import re, time
from openai import OpenAI

MODEL_ID = "gpt-4o"   # GPT-4o model


client = OpenAI(api_key=os.environ["OPENAI_API_KEY"])

def build_prompt(text: str, choices):
    return (
        "Analyze the sentiment of this financial text.\n"
        "Respond ONLY with one label from the Options.\n\n"
        f"Text: {text}\n"
        f"Options: {', '.join(map(str, choices))}\n"
        "Label:"
    )

def extract_label(model_text: str, choices):
    if not model_text:
        return None
    s = model_text.strip().lower()

    # exact match
    for c in choices:
        if s == str(c).strip().lower():
            return str(c)

    # whole-word match
    for c in choices:
        pat = r"\b" + re.escape(str(c).strip().lower()) + r"\b"
        if re.search(pat, s):
            return str(c)

    return None

def gpt4_predict(prompt: str, model: str = MODEL_ID, max_retries: int = 6, sleep_base: float = 1.0):
    # Responses API: :contentReference[oaicite:4]{index=4}
    last_err = None
    for attempt in range(max_retries):
        try:
            resp = client.responses.create(
                model=model,
                input=prompt,
            )
            text_out = getattr(resp, "output_text", None)

            usage = getattr(resp, "usage", None)
            usage_dict = None
            if usage is not None:
                usage_dict = {
                    "input_tokens": getattr(usage, "input_tokens", None),
                    "output_tokens": getattr(usage, "output_tokens", None),
                    "total_tokens": getattr(usage, "total_tokens", None),
                }

            meta = {
                "id": getattr(resp, "id", None),
                "usage": usage_dict,
            }
            return text_out, meta

        except Exception as e:
            last_err = str(e)
            time.sleep(sleep_base * (2 ** attempt))

    return None, {"error": last_err}


In [5]:
# =========================================
# Cell 5 — Run evaluation (full split by default) + resume support
# =========================================
import os, json, random, time
from tqdm import tqdm

# ---- controls ----
SEED = 42
MAX_SAMPLES = None        # None = FULL split; set an int for a quick test
SHUFFLE = True
SLEEP_BETWEEN_CALLS = 0.2

random.seed(SEED)

os.makedirs("text_responses", exist_ok=True)
os.makedirs("evaluation", exist_ok=True)

run_tag = f"{MODEL_ID}_{DATASET_NAME.replace('/','_')}_{SPLIT}"
raw_path = f"text_responses/{run_tag}.jsonl"

# resume: load completed row indices if file exists
done = set()
if os.path.exists(raw_path):
    with open(raw_path, "r", encoding="utf-8") as f:
        for line in f:
            try:
                done.add(json.loads(line)["row_index"])
            except:
                pass
print("Already completed:", len(done))

idxs = list(range(len(ds)))
if SHUFFLE:
    random.shuffle(idxs)
if MAX_SAMPLES is not None:
    idxs = idxs[:MAX_SAMPLES]

records = []
with open(raw_path, "a", encoding="utf-8") as f:
    for i in tqdm(idxs, desc="Evaluating"):
        if i in done:
            continue

        ex = ds[i]
        text = ex["text"]
        choices = ex["choices"]
        gold = int(ex["gold"])

        prompt = build_prompt(text, choices)
        model_out, meta = gpt4_predict(prompt)

        pred_label = extract_label(model_out, choices)
        pred_idx = choices.index(pred_label) if pred_label in choices else -1

        rec = {
            "row_index": i,
            "id": ex.get("id", None),
            "query": ex.get("query", None),
            "answer": ex.get("answer", None),
            "text": text,
            "choices": choices,
            "gold": gold,
            "model_response_raw": model_out,
            "predicted_label": pred_label,
            "predicted_index": pred_idx,
            "correct": (pred_idx == gold),
            "meta": meta,  # JSON-serializable
        }

        # write safely
        try:
            f.write(json.dumps(rec, ensure_ascii=False) + "\n")
        except TypeError:
            rec["meta"] = {"meta_str": str(meta)}
            f.write(json.dumps(rec, ensure_ascii=False) + "\n")

        records.append(rec)

        if SLEEP_BETWEEN_CALLS:
            time.sleep(SLEEP_BETWEEN_CALLS)

print("✅ Saved raw responses to:", raw_path)
print("New samples evaluated this run:", len(records))


Already completed: 0


Evaluating: 100%|██████████| 235/235 [04:16<00:00,  1.09s/it]

✅ Saved raw responses to: text_responses/gpt-4o_TheFinAI_flare-fiqasa_test.jsonl
New samples evaluated this run: 235


In [6]:
# =========================================
# Cell 6 — Compute metrics + save artifacts (loads full jsonl)
# =========================================
import pandas as pd
import numpy as np
from sklearn.metrics import accuracy_score, f1_score, confusion_matrix, classification_report
import json

all_rows = []
with open(raw_path, "r", encoding="utf-8") as f:
    for line in f:
        try:
            all_rows.append(json.loads(line))
        except:
            pass

df = pd.DataFrame(all_rows)

valid_mask = df["predicted_index"] >= 0
labels_sorted = sorted(set(df["gold"].tolist()))

acc_all = accuracy_score(df["gold"], df["predicted_index"])
acc_valid = accuracy_score(df.loc[valid_mask, "gold"], df.loc[valid_mask, "predicted_index"]) if valid_mask.any() else None

metrics = {
    "run_tag": run_tag,
    "model_id": MODEL_ID,
    "dataset": DATASET_NAME,
    "split": SPLIT,
    "num_samples": int(len(df)),
    "num_valid_predictions": int(valid_mask.sum()),
    "accuracy_all": float(acc_all),
    "accuracy_valid_only": (float(acc_valid) if acc_valid is not None else None),
    "f1_macro_valid_only": float(f1_score(df.loc[valid_mask, "gold"], df.loc[valid_mask, "predicted_index"],
                                         average="macro", labels=labels_sorted)) if valid_mask.any() else None,
    "f1_micro_valid_only": float(f1_score(df.loc[valid_mask, "gold"], df.loc[valid_mask, "predicted_index"],
                                         average="micro", labels=labels_sorted)) if valid_mask.any() else None,
    "f1_weighted_valid_only": float(f1_score(df.loc[valid_mask, "gold"], df.loc[valid_mask, "predicted_index"],
                                            average="weighted", labels=labels_sorted)) if valid_mask.any() else None,
}

cm = confusion_matrix(df.loc[valid_mask, "gold"], df.loc[valid_mask, "predicted_index"], labels=labels_sorted) if valid_mask.any() else None

metrics_path = f"evaluation/{run_tag}_metrics.json"
with open(metrics_path, "w", encoding="utf-8") as f:
    json.dump(
        {"metrics": metrics, "confusion_matrix_labels": labels_sorted, "confusion_matrix": (cm.tolist() if cm is not None else None)},
        f, ensure_ascii=False, indent=2
    )

preds_path = f"evaluation/{run_tag}_predictions.csv"
df.to_csv(preds_path, index=False)

print("✅ Metrics saved to:", metrics_path)
print("✅ Predictions saved to:", preds_path)

if valid_mask.any():
    print("\nClassification report (valid predictions only):")
    print(classification_report(df.loc[valid_mask, "gold"], df.loc[valid_mask, "predicted_index"], labels=labels_sorted))
else:
    print("No valid predictions to report.")


✅ Metrics saved to: evaluation/gpt-4o_TheFinAI_flare-fiqasa_test_metrics.json
✅ Predictions saved to: evaluation/gpt-4o_TheFinAI_flare-fiqasa_test_predictions.csv

Classification report (valid predictions only):
              precision    recall  f1-score   support

           0       0.85      0.82      0.83        76
           1       0.95      0.75      0.84       141
           2       0.22      0.61      0.32        18

    accuracy                           0.76       235
   macro avg       0.67      0.73      0.66       235
weighted avg       0.86      0.76      0.80       235

